In [ ]:
#NAME: Adebonojo Adesayo Oluwatosin
#STUDENT ID: X24248487
# install libraries
import pandas as pd
import json
import os
import psycopg2
import requests
%pip install pymongo
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
%pip install SQLAlchemy psycopg-binary
from sqlalchemy import create_engine
%pip install psycopg2-binary python-dotenv
from psycopg2.extras import execute_values
import matplotlib.pyplot as plt
%pip install seaborn
import seaborn as sns

In [ ]:
#  retrieve female json data from World bank API for all countries
indicator_female = "SE.ADT.LITR.FE.ZS"# Indicator for adult female literacy
country_all = "all" # all countrys
years = "2000:2023"

# api request  for female and return as json format
url_female_global = (
    f"https://api.worldbank.org/v2/country/{country_all}/indicator/{indicator_female}"
    f"?format=json&per_page=2000&date={years}")
# GET request
try:
    response_female_global = requests.get(url_female_global, timeout=10) # timeout after 10seconds
    response_female_global.raise_for_status() # if API requests returns an error
    print("Global FEMALE status code:",response_female_global.status_code)#quick check
    #coverts raw API into JSON
    data_female_global = response_female_global.json()
    female_global_meta = data_female_global[0] # meta data
    female_global_records = data_female_global[1] # actual records
    print("Global FEMALE total records:",len(female_global_records))#quick check
    print("Global FEMALE meta:", female_global_meta)#quick check
except requests.exceptions.Timeout:
  print("Error: Request timed out. Check internet connection.")
except requests.exceptions.HTTPError as ht_error:
  print(f"HTTP Error: {ht_error}")
except ValueError:
  print(f"Error: Invalid JSON")  
except Exception as e:
  print(f"An unexpected error occurred: {e}")

In [ ]:
# Flatten  records into a DataFrame
raw_female_global_df = pd.json_normalize(female_global_records)
raw_female_global_df.head()#quick check

In [ ]:
#  retrieve male json data from World bank API for all countries
indicator_male = "SE.ADT.LITR.MA.ZS"# Indicator for adult male literacy
country_all = "all" # all countrys
years = "2000:2023"

# api request  for male and return as json format
url_male_global = (
    f"https://api.worldbank.org/v2/country/{country_all}/indicator/{indicator_male}"
    f"?format=json&per_page=2000&date={years}")
# GET request
try:
    response_male_global = requests.get(url_male_global, timeout=20) #timeout after 20seconds
    response_male_global.raise_for_status() # if API requests returns an error
    print("Global MALE status code:",response_male_global.status_code)#quick check
    #coverts raw API into JSON
    data_male_global = response_male_global.json()
    male_global_meta = data_male_global[0] # meta data
    male_global_records = data_male_global[1] # actual records
    print("Global MALE total records:",len(male_global_records))#quick check
    print("Global MALE meta:", male_global_meta)#quick check
except requests.exceptions.Timeout:
  print("Error: Request timed out. Check internet connection.")
except requests.exceptions.HTTPError as ht_error:
  print(f"HTTP Error: {ht_error}")
except ValueError:
  print(f"Error: Invalid JSON")  
except Exception as e:
  print(f"An unexpected error occurred: {e}")

In [ ]:
# Flatten  records into a DataFrame
raw_male_global_df = pd.json_normalize(male_global_records)
raw_male_global_df.head()#quick check

In [ ]:
# Load MongoDB login details from data.env and connect to the local Docker database
load_dotenv("../data.env", override=True)

MONGO_USERNAME = os.getenv("MONGO_USERNAME")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")
MONGO_HOST = os.getenv("MONGO_HOST", "localhost")
MONGO_PORT = int(os.getenv("MONGO_PORT", "27017"))
MONGO_DB = os.getenv("MONGO_DB")

client = MongoClient(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    authSource="admin"
)

try:
    client.admin.command("ping")
    print("Connected to MongoDB")
except Exception as e:
    print("Connection Error:", e)

db = client[MONGO_DB]

# Collections for GLOBAL data
female_global_coll = db["wb_literacy_female_global"] # create female global collection
male_global_coll   = db["wb_literacy_male_global"] # create male global collection

# Clear old documents so there is no duplicate
female_global_coll.delete_many({})
male_global_coll.delete_many({})

# Insert new documents
if female_global_records:
    result_female_global = female_global_coll.insert_many(female_global_records)
    print("Inserted GLOBAL female docs:",len(result_female_global.inserted_ids))#quick check for inserted global female docs

if male_global_records:
    result_male_global = male_global_coll.insert_many(male_global_records)
    print("Inserted GLOBAL male docs:",len(result_male_global.inserted_ids))#quick check for inserted global male docs

In [ ]:
#  retrieve female json data from World bank API for all countries
indicator_female = "SE.ADT.LITR.FE.ZS"
country_nga = "NGA"          # Nigeria ISO3 code
years = "2000:2023"

# api request  for nigeria female and return as json format
url_female_nga = (
    f"https://api.worldbank.org/v2/country/{country_nga}/indicator/{indicator_female}"
    f"?format=json&per_page=2000&date={years}")
# GET request
try:
    response_female_nga = requests.get(url_female_nga, timeout=10)
    response_female_nga.raise_for_status() # if API requests returns an error
    print("Global FEMALE status code:",response_female_nga.status_code)
    #coverts raw API into JSON
    data_female_nga = response_female_nga.json()
    female_nga_meta = data_female_nga[0] # meta data
    female_nga_records = data_female_nga[1] # actual records
    print("Global FEMALE total records:",len(female_nga_records))
    print("Global FEMALE meta:", female_nga_meta)
except requests.exceptions.Timeout:
  print("Error: Request timed out. Check internet connection.")
except requests.exceptions.HTTPError as ht_error:
  print(f"HTTP Error: {ht_error}")
except ValueError:
  print(f"Error: Invalid JSON")  
except Exception as e:
  print(f"An unexpected error occurred: {e}")

In [ ]:
# Flatten female nigeria to DataFrame
raw_female_ng_df = pd.json_normalize(female_nga_records)
raw_female_ng_df.head()#quick check

In [ ]:
#  retrieve male json data from World bank API for all countries
indicator_male = "SE.ADT.LITR.MA.ZS"
country_nga = "NGA"          # Nigeria ISO3 code
years = "2000:2023"

# api request  for nigeria male and return as json format
url_male_nga = (
    f"https://api.worldbank.org/v2/country/{country_nga}/indicator/{indicator_male}"
    f"?format=json&per_page=2000&date={years}")
# GET request
try:
    response_male_nga = requests.get(url_male_nga, timeout=30)
    response_male_nga.raise_for_status() # if API requests returns an error
    print("Global MALE status code:",response_male_nga.status_code)
    #coverts raw API into JSON
    data_male_nga = response_male_nga.json()
    male_nga_meta = data_male_nga[0] # meta data
    male_nga_records = data_male_nga[1] # actual records
    print("Global MALE total records:",len(male_nga_records))
    print("Global MALE meta:", male_nga_meta)
except requests.exceptions.Timeout:
  print("Error: Request timed out. Check internet connection.")
except requests.exceptions.HTTPError as ht_error:
  print(f"HTTP Error: {ht_error}")
except ValueError:
  print(f"Error: Invalid JSON")  
except Exception as e:
  print(f"An unexpected error occurred: {e}")

In [ ]:
# Flatten nigeria male to DataFrame
raw_male_ng_df = pd.json_normalize(male_nga_records)
raw_male_ng_df.head()#quick check

In [ ]:
print("Global female rows:",len(raw_female_global_df))#quick check
print("Global male rows:",len(raw_male_global_df))#quick check

In [ ]:
# store  nigeria data in mongo db
female_nga_coll = db["wb_literacy_female_nga"] # create female nigeria collections
male_nga_coll   = db["wb_literacy_male_nga"] # create male nigerian collections

# Clear old Nigeria documents so that there is no duplicate
female_nga_coll.delete_many({})
male_nga_coll.delete_many({})
# insert new documents
if female_nga_records:
    result_female_nga = female_nga_coll.insert_many(female_nga_records)
    print("Inserted NIGERIA female docs:",len(result_female_nga.inserted_ids))#quick check

if male_nga_records:
    result_male_nga = male_nga_coll.insert_many(male_nga_records)
    print("Inserted NIGERIA male docs:",len(result_male_nga.inserted_ids))#quick check

In [ ]:
# select important colunms and clean types for nigerain female
female_df = raw_female_ng_df[[
    "country.value",    
    "countryiso3code",  
    "date",             
    "value",            
    "indicator.id",     
    "indicator.value"   
]].copy()# select the colunms that will be used and store in a dataframe

# Rename columns 
female_df.rename(columns={
    "country.value": "country",
    "countryiso3code": "country_code",
    "date": "year",
    "value": "literacy_rate_female",
    "indicator.id": "indicator_id_female",
    "indicator.value": "indicator_value_female"
},inplace=True) # rename the colunms for readability

# convert year to integer and literacy_rate to numeric
female_df["year"] = pd.to_numeric(female_df["year"],errors="coerce").astype("Int64") # convert the year in female datafram to integer
female_df["literacy_rate_female"] = pd.to_numeric(
    female_df["literacy_rate_female"],errors="coerce")
# Sort rows by year so that interpolation works
female_df.sort_values(["year"],inplace=True) # sort the rows by year

# flag the reported values
female_df["is_reported_female"] = female_df["literacy_rate_female"].notna().astype(int) # create a new colunm is_reported_female for reported values in literacy_rate_female

# Interpolate missing literacy values with matching index
female_df["literacy_rate_interp_female"] = (female_df["literacy_rate_female"].interpolate(method="linear",limit_direction="both")) # interpolation through literacy_Rate_female to get forecasted values

# Reset index 
female_df.reset_index(drop=True, inplace=True)
female_df.head()#quick check

In [ ]:
# select important colunms and clean types for nigerai male
male_df = raw_male_ng_df[[
    "country.value",    
    "countryiso3code",  
    "date",             
    "value",            
    "indicator.id",     
    "indicator.value"   
]].copy()#select the colunms that will be used and store in a dataframe

# Rename columns 
male_df.rename(columns={
    "country.value": "country",
    "countryiso3code": "country_code",
    "date": "year",
    "value": "literacy_rate_male",
    "indicator.id": "indicator_id_male",
    "indicator.value": "indicator_value_male"
}, inplace=True)# rename the colunms for readability

# convert year to integer and literacy_rate to numeric
male_df["year"] = pd.to_numeric(male_df["year"],errors="coerce").astype("Int64")# convert the year in female datafram to integer
male_df["literacy_rate_male"] = pd.to_numeric(
    male_df["literacy_rate_male"],errors="coerce")
# Sort rows by year so that interpolation works
male_df.sort_values(["year"],inplace=True)# sort the rows by year

# flag the reported values for males in nigeria
male_df["is_reported_male"] = male_df["literacy_rate_male"].notna().astype(int)# create a new colunm is_reported_male for reported values in literacy_rate_female

# Interpolate missing literacy values with matching index
male_df["literacy_rate_interp_male"] = (male_df["literacy_rate_male"].interpolate(method="linear",limit_direction="both"))# interpolation through literacy_Rate_male to get forecasted values


# Reset index 
male_df.reset_index(drop=True,inplace=True)
male_df.head() #quick check

In [ ]:
# merge both female and male
# Merge on country, country_code, and year
combined_df = pd.merge(
    female_df,
    male_df,
    on=["country","country_code","year"],
    how="inner",
    suffixes=("_female","_male")  # adds to end of the columns
) # merge female and male dataframe 

# Create gender gap by subtracting the interpolation female from the male
combined_df["gender_gap"] = (
    combined_df["literacy_rate_interp_male"] -
    combined_df["literacy_rate_interp_female"]
)
# Clean up columns 
combined_df = combined_df[[
    "country",
    "country_code",
    "year",
    # female nigeria original plus the interpolated
    "literacy_rate_female",
    "literacy_rate_interp_female",
    "is_reported_female",
    # male nigeria original plus the interpolated
    "literacy_rate_male",
    "literacy_rate_interp_male",
    "is_reported_male",
    # gender gap
    "gender_gap",
    # indicator metadata
    "indicator_id_female",
    "indicator_value_female",
    "indicator_id_male",
    "indicator_value_male"
]]
combined_df.head()#quick check


In [ ]:
# EDA on combined df
full_df = combined_df.copy()
display(full_df[[
    "literacy_rate_female",
    "literacy_rate_interp_female",
    "literacy_rate_male",
    "literacy_rate_interp_male",
    "gender_gap"
]].describe()) # descrptive analysis of data

In [ ]:
# missing values and interpolation
print("Missing values per column:")
print(full_df.isna().sum()) # missing values for each colunm

print("\nReported vs interpolated (FEMALE):")
print(full_df["is_reported_female"].value_counts()) # count reported and interpolated values for female

print("\nReported vs interpolated (MALE):")
print(full_df["is_reported_male"].value_counts()) # count reported and interpolated values for male

In [ ]:
# Clean country code 
female_df["country_code"] = female_df["country_code"].astype(str).str.strip() # clean country_code for female
male_df["country_code"] = male_df["country_code"].astype(str).str.strip()#clean country_code for male 
# Filter Nigeria 
female_ng = female_df[female_df["country_code"] == "NGA"].copy() #filter for nigeria
male_ng   = male_df[male_df["country_code"] == "NGA"].copy()#filter for nigeriaa
print("Female Nigeria rows:",female_ng.shape)
print("Male Nigeria rows:",male_ng.shape)

In [ ]:
# merge female and male nigeria data
# merge on country code and year
combined_df = pd.merge(female_ng,
    male_ng,
    on=["country_code","year"],
    how="inner",
    suffixes=("_female","_male")) # adds to end of the columns
# input gender gap column which is male minus female literacy
combined_df["gender_gap"] = (combined_df["literacy_rate_interp_male"] - combined_df["literacy_rate_interp_female"]
)
print("Combined dataset shape:",combined_df.shape)
combined_df.head()


In [ ]:
combined_df.describe() # statistics summary

In [ ]:
print(combined_df.isna().sum()) # missing values per colunms

In [ ]:
# missing female literacy rows and values and years where interpolation was applied
female_interp_rows = combined_df[combined_df["is_reported_female"] == 0][
    ["year","literacy_rate_female","literacy_rate_interp_female"]
]
female_interp_rows

In [ ]:
# missing male literacy rows and values and years where interpolation was applied
male_interp_rows = combined_df[combined_df["is_reported_male"] == 0][
    ["year", "literacy_rate_male","literacy_rate_interp_male"]
]
male_interp_rows

In [ ]:
# Save Trend plot as PNG
os.makedirs("../visualisations",exist_ok=True) # create visualisation folder in directory, if it exists nothing should happen
plt.figure(figsize=(10,5))
plt.plot(combined_df["year"],combined_df["literacy_rate_interp_female"],marker="o",label="Female Literacy (Interpolated)")
plt.plot(combined_df["year"],combined_df["literacy_rate_interp_male"],marker="o",label="Male Literacy (Interpolated)")
plt.title("Interpolated Male vs Female Literacy Rate in Nigeria (2000-2023)", fontsize=14, fontweight="bold")
plt.xlabel("Year",fontsize=12)
plt.ylabel("Literacy Rate",fontsize=12)
plt.grid(True)
plt.savefig("../visualisations/sayo_male_vs_female_trend_plot.png",dpi=300,bbox_inches="tight") # save as trend_plot in visualisation folder
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(combined_df["year"],combined_df["gender_gap"],marker="o",label="Gender Literacy Gap")
plt.title("Interpolated Gender Literacy Gap in Nigeria (2000-2023)", fontsize=14, fontweight="bold")
plt.xlabel("Year",fontsize=12)
plt.ylabel("Gender Literacy Gap (Male - Female)",fontsize=12)
plt.grid(True)
plt.savefig("../visualisations/sayo_gender_gap_trend_plot.png",dpi=300,bbox_inches="tight") # save as gender_gap_trend_plot in visualisation folder
plt.show()


In [ ]:
# histogram of avaliable literacy rate
plt.figure(figsize=(8,4))
sns.histplot(combined_df["gender_gap"],bins=6,kde=True)
plt.title("Distribution of Interpolated Gender Literacy Gap in Nigeria (2000-2023)", fontsize=14, fontweight="bold")
plt.xlabel("Gender Literacy Gap (Male - Female)",fontsize=12)
plt.ylabel("Frequency",fontsize=12)
plt.grid(True)
plt.savefig("../visualisations/sayo_gender_gap_histogram.png",dpi=300,bbox_inches="tight") # save as gender_gap_histogram in visualisation folder
plt.show()


In [ ]:
#  missing data plot
# columns used for the heatmap
mis_cols = [
    "literacy_rate_female",
    "literacy_rate_interp_female",
    "literacy_rate_male",
    "literacy_rate_interp_male"
]
plt.figure(figsize=(6,4))
sns.heatmap(combined_df[mis_cols].isna(),cmap="Reds",cbar=False)
plt.title("Missing Data Heatmap",fontsize=14,fontweight="bold")
plt.xlabel("Columns",fontsize=12)
plt.ylabel("Rows",fontsize=12)
plt.savefig("../visualisations/sayo_missing_data_heatmap.png",dpi=300,bbox_inches="tight") # save as missing_data_heatmap in visualisation folder
plt.show()


In [ ]:
# connect to postgresql
load_dotenv("../data.env", override=True) # loads data.env into current directory

conn = None
try:
    conn = psycopg2.connect(
        user=os.getenv("DATABASE_USERNAME"),
        password=os.getenv("DATABASE_PASSWORD"),
        host=os.getenv("DATABASE_HOST"),
        port=os.getenv("DATABASE_PORT"),
        dbname=os.getenv("DATABASE_NAME"),
    ) # runs the connection to postgres
    print("Connection to:", os.getenv("DATABASE_NAME"))
except Exception as e:
    print("Connection error:", e)
finally:
    if conn:
        conn.close()


In [ ]:
# select only columns for sql storage
sql_cols = [
    "country_code",
    "year",
    "literacy_rate_female",
    "literacy_rate_interp_female",
    "literacy_rate_male",
    "literacy_rate_interp_male",
    "gender_gap"
] # colunms for sql
gender_sql_df = combined_df[sql_cols].copy() # create dataframe for male and female combined dataframe with selected colunms
print("gender_sql_df shape:",gender_sql_df.shape)
gender_sql_df.head()#quick check

In [ ]:
ddl = """
DROP TABLE IF EXISTS gender_literacy_final;

CREATE TABLE gender_literacy_final (
    country_code                 TEXT,
    year                         INT,
    literacy_rate_female         FLOAT,
    literacy_rate_interp_female  FLOAT,
    literacy_rate_male           FLOAT,
    literacy_rate_interp_male    FLOAT,
    gender_gap                   FLOAT
);
""" # drops the taable if it exist and create a new table
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(ddl) # execute the ddl
conn.close()
print("gender_literacy_final table created successfully.")


In [ ]:
# convert DataFrame rows to list of tuples

records = [
    (
        str(row["country_code"]) if row["country_code"] is not None else None,# Convert  to a string
        int(row["year"]), # covert to  integer
        float(row["literacy_rate_female"]) if not pd.isna(row["literacy_rate_female"]) else None,#Convert to float
        float(row["literacy_rate_interp_female"]) if not pd.isna(row["literacy_rate_interp_female"]) else None,#Convert to float
        float(row["literacy_rate_male"]) if not pd.isna(row["literacy_rate_male"]) else None,#Convert to float
        float(row["literacy_rate_interp_male"]) if not pd.isna(row["literacy_rate_interp_male"]) else None,#Convert to float
        float(row["gender_gap"]) if not pd.isna(row["gender_gap"]) else None#Convert to float
    )
    for _,row in gender_sql_df.iterrows()]# turns each row to a tuple and turn each row value into  python datatype


In [ ]:
insert_sql = """
INSERT INTO gender_literacy_final (
    country_code,
    year,
    literacy_rate_female,
    literacy_rate_interp_female,
    literacy_rate_male,
    literacy_rate_interp_male,
    gender_gap
)
VALUES (%s, %s, %s, %s, %s, %s, %s);
""" # insert coluns into gender_literacy_final table
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
) # establish the connection
with conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql,records) # insert the rows into the database
conn.close()
print("Inserted",len(records),"rows into gender_literacy_final.") # quick check 

In [ ]:
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)

df_check = pd.read_sql("SELECT * FROM gender_literacy_final ORDER BY year LIMIT 10;",conn) # select all colunms from table sort by year and show first 10 rows
conn.close()

df_check#quick check

In [ ]:
                                                            ####### Group Integration##############

In [ ]:
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)# establish connection to postgresql

df_lit = pd.read_sql("SELECT * FROM gender_literacy_final;",conn)# Load the gender_literacry_final from PostgreSQL to DataFrame
df_un = pd.read_sql("SELECT * FROM un_nigeria_wide",conn)# Load the un_nigeria_wide from PostgreSQL to DataFrame
df_enr = pd.read_sql("""
SELECT
    country_code,
    country_name,
    year,
    primary_female,
    primary_male,
    secondary_female,
    secondary_male,
    primary_gap_female_minus_male,
    secondary_gap_female_minus_male
FROM wb_gender_enrolment
WHERE country_code = 'NGA'
""", conn)

In [ ]:
print(df_enr["country_code"].unique())
print(df_enr["country_name"].unique())
print("Rows:", len(df_enr))

display(
    df_enr[
        ["country_code", "country_name", "year",
         "primary_female", "primary_male"]
    ].sort_values("year").head(10)
)

In [ ]:
print("Duplicate years:", df_enr["year"].duplicated().sum())

In [ ]:
merge_1 = pd.merge(
    df_lit,
    df_enr,
    on=["country_code", "year"],
    how="left"
)

full_merged = pd.merge(
    merge_1,
    df_un,
    on=["country_code", "year"],
    how="left"
)

In [ ]:
print("Rows:", len(full_merged))
print("Duplicate years:", full_merged["year"].duplicated().sum())

display(
    full_merged[
        [
            "year",
            "literacy_rate_interp_female",
            "literacy_rate_interp_male",
            "primary_female",
            "primary_male",
            "un_primary_female",
            "un_primary_male"
        ]
    ].head(15)
)

In [ ]:
merge_1 = pd.merge(df_lit,df_enr,on=["country_code","year"],how="left") # merge gender_literacy_final and un_nigeria_wide on country_code and year to create first merge
merge_1.head()#quick check

In [ ]:
full_merged = pd.merge(merge_1,df_un,on=["country_code","year"],how="left")# merge the first merge and un_nigeria_wide on country_code and year to create a final merged
full_merged.head()#quick check

In [ ]:
ddl_merged = """
DROP TABLE IF EXISTS education_final_merged;

CREATE TABLE education_final_merged(
    country_code CHAR(3) NOT NULL,
    year INT NOT NULL,
    country_name TEXT,
    literacy_rate_female NUMERIC(6,3),
    literacy_rate_interp_female NUMERIC(6,3),
    literacy_rate_male NUMERIC(6,3),
    literacy_rate_interp_male NUMERIC(6,3),
    gender_gap NUMERIC(7,3),
    primary_female NUMERIC(8,3),
    primary_male NUMERIC(8,3),
    secondary_female NUMERIC(8,3),
    secondary_male NUMERIC(8,3),
    primary_gap_female_minus_male NUMERIC(8,3),
    secondary_gap_female_minus_male NUMERIC(8,3),
    un_primary_male NUMERIC(10,3),
    un_primary_female NUMERIC(10,3),
    un_lower_secondary_male NUMERIC(10,3),
    un_lower_secondary_female NUMERIC(10,3),
    un_upper_secondary_male NUMERIC(10,3),
    un_upper_secondary_female NUMERIC(10,3)
);
""" # create ddl statement and create and  remove the final table if it exists 
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(ddl_merged)#execute ddl statement
conn.close()
print("education_final_merged created successfully.")#quick check

In [ ]:
records_merged = [
    (
        str(row["country_code"]) if pd.notna(row["country_code"]) else None,#Convert to string
        int(row["year"]) if pd.notna(row["year"]) else None,#Convert to integer
        str(row["country_name"]) if pd.notna(row["country_name"]) else None,#Convert to string
        float(row["literacy_rate_female"]) if pd.notna(row["literacy_rate_female"]) else None,#Convert to float
        float(row["literacy_rate_interp_female"]) if pd.notna(row["literacy_rate_interp_female"]) else None,#Convert to float
        float(row["literacy_rate_male"]) if pd.notna(row["literacy_rate_male"]) else None,#Convert to float
        float(row["literacy_rate_interp_male"]) if pd.notna(row["literacy_rate_interp_male"]) else None,#Convert to float
        float(row["gender_gap"]) if pd.notna(row["gender_gap"]) else None,#Convert to float
        float(row["primary_female"]) if pd.notna(row["primary_female"]) else None,#Convert to float
        float(row["primary_male"]) if pd.notna(row["primary_male"]) else None,#Convert to float
        float(row["secondary_female"]) if pd.notna(row["secondary_female"]) else None,#Convert to float
        float(row["secondary_male"]) if pd.notna(row["secondary_male"]) else None,#Convert to float
        float(row["primary_gap_female_minus_male"]) if pd.notna(row["primary_gap_female_minus_male"]) else None,#Convert to float
        float(row["secondary_gap_female_minus_male"]) if pd.notna(row["secondary_gap_female_minus_male"]) else None,#Convert to float
        float(row["un_primary_male"]) if pd.notna(row["un_primary_male"]) else None,#Convert to float
        float(row["un_primary_female"]) if pd.notna(row["un_primary_female"]) else None,#Convert to float
        float(row["un_lower_secondary_male"]) if pd.notna(row["un_lower_secondary_male"]) else None,#Convert to float
        float(row["un_lower_secondary_female"]) if pd.notna(row["un_lower_secondary_female"]) else None,#Convert to float
        float(row["un_upper_secondary_male"]) if pd.notna(row["un_upper_secondary_male"]) else None,#Convert to float
        float(row["un_upper_secondary_female"]) if pd.notna(row["un_upper_secondary_female"]) else None#Convert to float
    )
    for _,row in full_merged.iterrows()
]# turns each row to a tuple and turn each row value into a  python datatype


In [ ]:
insert_sql_merged = """
INSERT INTO education_final_merged(
    country_code, year, country_name,
    literacy_rate_female, literacy_rate_interp_female,
    literacy_rate_male, literacy_rate_interp_male,
    gender_gap,
    primary_female, primary_male,
    secondary_female, secondary_male,
    primary_gap_female_minus_male,
    secondary_gap_female_minus_male,
    un_primary_male, un_primary_female,
    un_lower_secondary_male, un_lower_secondary_female,
    un_upper_secondary_male, un_upper_secondary_female
)
VALUES (
    %s, %s, %s,
    %s, %s,
    %s, %s,
    %s,
    %s, %s,
    %s, %s,
    %s,
    %s,
    %s, %s,
    %s, %s,
    %s, %s
);
"""# insert coluns into education_final_merged table
conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)
with conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql_merged,records_merged)# insert the rows into the database
conn.close()
print("Inserted", len(records_merged), "rows into education_final_merged.")#qucik check

In [ ]:

conn = psycopg2.connect(
    user=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=os.getenv("DATABASE_PORT"),
    dbname=os.getenv("DATABASE_NAME")
)

check_final = pd.read_sql("""
SELECT
    year,
    country_code,
    primary_female,
    primary_male,
    un_primary_female,
    un_primary_male
FROM education_final_merged
ORDER BY year;
""", conn)

conn.close()

print("Rows:", len(check_final))
print("Duplicate years:", check_final["year"].duplicated().sum())
display(check_final)

In [ ]:
df_filtered = full_merged[full_merged["year"] <= 2013].copy() #filter out where year is less than 2013

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    df_filtered["year"],
    df_filtered["literacy_rate_interp_female"],
    marker="o",
    label="Female Literacy - Interpolated (World Bank)"
)

plt.plot(
    df_filtered["year"],
    df_filtered["primary_female"],
    marker="o",
    label="Female Primary Enrolment (World Bank)"
)

plt.title("Female Literacy vs Female Primary Enrolment in Nigeria (2000-2013)")
plt.xlabel("Year")
plt.ylabel("Rate (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    "../visualisations/Female Literacy vs Female Primary Enrolment in Nigeria (2000-2013).png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    df_filtered["year"],
    df_filtered["literacy_rate_interp_male"],
    marker="o",
    label="Male Literacy - Interpolated (World Bank)"
)

plt.plot(
    df_filtered["year"],
    df_filtered["primary_male"],
    marker="o",
    label="Male Primary Enrolment (World Bank)"
)

plt.title("Male Literacy vs Male Primary Enrolment in Nigeria (2000-2013)")
plt.xlabel("Year")
plt.ylabel("Rate (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    "../visualisations/Male Literacy vs Male Primary Enrolment in Nigeria (2000-2013).png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# filter out the un_primary_male has nan values for analysis
male_filtered = full_merged[
    full_merged["un_primary_male"].notna()
].copy()

display(
    male_filtered[
        ["year", "literacy_rate_interp_male",
         "primary_male", "un_primary_male"]
    ]
)

In [ ]:
male_filtered.head()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    male_filtered["year"],
    male_filtered["literacy_rate_interp_male"],
    marker="o",
    label="Male Literacy - Interpolated (World Bank)"
)

plt.plot(
    male_filtered["year"],
    male_filtered["un_primary_male"],
    marker="o",
    label="Male Primary Enrolment (United Nations)"
)

plt.plot(
    male_filtered["year"],
    male_filtered["primary_male"],
    marker="o",
    label="Male Primary Enrolment (World Bank)"
)

plt.title("Male Literacy and Primary Enrolment in Nigeria")
plt.xlabel("Year")
plt.ylabel("Rate (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    "../visualisations/Male Literacy vs Male Primary Enrolment vs Male Primary(UN) in Nigeria.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
female_filtered = full_merged[
    full_merged["un_primary_female"].notna()
].copy()

display(
    female_filtered[
        ["year", "literacy_rate_interp_female",
         "primary_female", "un_primary_female"]
    ]
)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    female_filtered["year"],
    female_filtered["literacy_rate_interp_female"],
    marker="o",
    label="Female Literacy - Interpolated (World Bank)"
)

plt.plot(
    female_filtered["year"],
    female_filtered["un_primary_female"],
    marker="o",
    label="Female Primary Enrolment (United Nations)"
)

plt.plot(
    female_filtered["year"],
    female_filtered["primary_female"],
    marker="o",
    label="Female Primary Enrolment (World Bank)"
)

plt.title("Female Literacy and Primary Enrolment in Nigeria")
plt.xlabel("Year")
plt.ylabel("Rate (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    "../visualisations/Female Literacy vs Female Primary Enrolment vs Female Primary(UN) in Nigeria.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("Final rows:", len(full_merged))
print("Duplicate years:", full_merged["year"].duplicated().sum())
print("Years:", full_merged["year"].min(), "-", full_merged["year"].max())

print(
    full_merged[
        ["un_primary_female", "un_primary_male"]
    ].notna().sum()
)

In [ ]:
from pathlib import Path

viz_dir = Path("../visualisations")

rename_map = {
    "sayo_male_vs_female_trend_plot.png":
        "sayo_interpolated_male_vs_female_literacy_2000_2023.png",

    "sayo_gender_gap_trend_plot.png":
        "sayo_interpolated_gender_literacy_gap_trend_2000_2023.png",

    "sayo_gender_gap_histogram.png":
        "sayo_interpolated_gender_literacy_gap_distribution_2000_2023.png",

    "sayo_missing_data_heatmap.png":
        "sayo_literacy_missing_data_heatmap_2000_2023.png",

    "Female Literacy vs Female Primary Enrolment in Nigeria (2000-2013).png":
        "sayo_female_literacy_vs_primary_enrolment_2000_2013.png",

    "Male Literacy vs Male Primary Enrolment in Nigeria (2000-2013).png":
        "sayo_male_literacy_vs_primary_enrolment_2000_2013.png",

    "Female Literacy vs Female Primary Enrolment vs Female Primary(UN) in Nigeria.png":
        "sayo_female_literacy_un_vs_world_bank_primary_enrolment.png",

    "Male Literacy vs Male Primary Enrolment vs Male Primary(UN) in Nigeria.png":
        "sayo_male_literacy_un_vs_world_bank_primary_enrolment.png"
}

for old_name, new_name in rename_map.items():
    old_file = viz_dir / old_name
    new_file = viz_dir / new_name

    if old_file.exists():
        old_file.rename(new_file)
        print(f"Renamed: {old_name} -> {new_name}")
    elif new_file.exists():
        print(f"Already renamed: {new_name}")
    else:
        print(f"Not found: {old_name}")